# Chapter 8.6: Item Understanding

## Learning Objectives

By the end of this notebook, you will be able to:

1. Build multimodal item embeddings combining text, image, and metadata features
2. Implement automated content tagging and attribute extraction
3. Design item lifecycle management (new, trending, declining items)
4. Understand ItemSage (Meta) multi-task item embeddings for downstream tasks
5. Build item graphs capturing co-purchase, co-view, and substitute/complement relationships
6. Create a complete multimodal item understanding pipeline
7. Evaluate item representations on retrieval tasks

## Prerequisites

- Chapter 8.4: ID vs ID-free systems
- Basic understanding of text and image representations
- Familiarity with graph concepts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part8/chapter_8.6_item_understanding.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/hideak1/rec_system/raw/main/notebooks/part8/chapter_8.6_item_understanding.ipynb)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8')
print(f"PyTorch version: {torch.__version__}")

## 1. Multimodal Item Embeddings

Modern items are described by multiple modalities:
- **Text**: title, description, reviews
- **Image**: product photos, thumbnails
- **Metadata**: category, brand, price, attributes
- **Behavioral**: click rates, conversion rates

A multimodal item encoder fuses these:

$$\mathbf{e}_i = \text{Fusion}(f_{\text{text}}(t_i), f_{\text{image}}(v_i), f_{\text{meta}}(m_i), f_{\text{behav}}(b_i))$$

Common fusion strategies:
1. **Late fusion**: Encode each modality separately, then concatenate + MLP
2. **Cross-modal attention**: Modalities attend to each other
3. **Gated fusion**: Learned gates determine each modality's contribution

> **\U0001f4a1 Concept:** The key insight from production systems is that **behavioral features dominate** for popular items, while **content features matter most** for cold-start items. A good fusion mechanism adapts accordingly.

In [ ]:
@dataclass
class ItemData:
    """Synthetic item with multimodal features."""
    item_id: int
    text_embedding: np.ndarray  # Simulated text encoder output
    image_embedding: np.ndarray  # Simulated image encoder output
    category: int
    price: float
    brand: int
    num_reviews: int
    avg_rating: float
    click_rate: float
    conversion_rate: float


def generate_synthetic_items(num_items: int, num_categories: int = 10, num_brands: int = 50) -> List[ItemData]:
    items = []
    for i in range(num_items):
        cat = np.random.randint(0, num_categories)
        brand = np.random.randint(0, num_brands)
        
        # Text embedding: correlated with category
        text_base = np.zeros(64)
        text_base[cat*6:(cat+1)*6] = 1.0
        text_emb = text_base + np.random.randn(64) * 0.3
        
        # Image embedding: correlated with category and price
        price = np.random.exponential(50) + 5
        img_base = np.zeros(64)
        img_base[cat*6:(cat+1)*6] = 0.8
        img_emb = img_base + np.random.randn(64) * 0.4
        
        items.append(ItemData(
            item_id=i,
            text_embedding=text_emb,
            image_embedding=img_emb,
            category=cat,
            price=price,
            brand=brand,
            num_reviews=max(0, int(np.random.exponential(50))),
            avg_rating=np.clip(np.random.normal(3.5, 1.0), 1.0, 5.0),
            click_rate=np.random.beta(2, 8),
            conversion_rate=np.random.beta(1, 20),
        ))
    return items

# Generate items
num_items = 5000
num_categories = 10
num_brands = 50
items = generate_synthetic_items(num_items, num_categories, num_brands)

print(f"Generated {len(items)} items")
print(f"Sample item: category={items[0].category}, price=${items[0].price:.2f}, "
      f"reviews={items[0].num_reviews}, rating={items[0].avg_rating:.1f}")

In [ ]:
class MultimodalItemEncoder(nn.Module):
    """Multimodal item encoder with gated fusion."""
    
    def __init__(
        self,
        text_dim: int = 64,
        image_dim: int = 64,
        num_categories: int = 10,
        num_brands: int = 50,
        metadata_dim: int = 4,  # price, reviews, rating, ctr
        embedding_dim: int = 32,
        output_dim: int = 64,
    ):
        super().__init__()
        
        # Modality encoders
        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, 64), nn.ReLU(), nn.Linear(64, output_dim)
        )
        self.image_encoder = nn.Sequential(
            nn.Linear(image_dim, 64), nn.ReLU(), nn.Linear(64, output_dim)
        )
        
        # Categorical embeddings
        self.category_emb = nn.Embedding(num_categories, embedding_dim)
        self.brand_emb = nn.Embedding(num_brands, embedding_dim)
        
        # Metadata encoder
        self.meta_encoder = nn.Sequential(
            nn.Linear(metadata_dim + embedding_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim),
        )
        
        # Gated fusion
        self.gate_network = nn.Sequential(
            nn.Linear(output_dim * 3, 64),
            nn.ReLU(),
            nn.Linear(64, 3),  # 3 modalities
            nn.Softmax(dim=-1),
        )
        
        # Final projection
        self.output_proj = nn.Linear(output_dim, output_dim)
    
    def forward(
        self,
        text_features: torch.Tensor,
        image_features: torch.Tensor,
        categories: torch.Tensor,
        brands: torch.Tensor,
        metadata: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Encode each modality
        text_repr = self.text_encoder(text_features)
        image_repr = self.image_encoder(image_features)
        
        cat_emb = self.category_emb(categories)
        brand_emb = self.brand_emb(brands)
        meta_input = torch.cat([metadata, cat_emb, brand_emb], dim=-1)
        meta_repr = self.meta_encoder(meta_input)
        
        # Gated fusion
        combined = torch.cat([text_repr, image_repr, meta_repr], dim=-1)
        gates = self.gate_network(combined)  # (batch, 3)
        
        stacked = torch.stack([text_repr, image_repr, meta_repr], dim=1)  # (batch, 3, dim)
        fused = (stacked * gates.unsqueeze(-1)).sum(dim=1)  # (batch, dim)
        
        output = self.output_proj(fused)
        return output, gates

# Prepare data tensors
text_features = torch.tensor(np.array([it.text_embedding for it in items]), dtype=torch.float32)
image_features = torch.tensor(np.array([it.image_embedding for it in items]), dtype=torch.float32)
categories = torch.tensor([it.category for it in items])
brands = torch.tensor([it.brand for it in items])
metadata = torch.tensor([
    [it.price / 100, np.log1p(it.num_reviews), it.avg_rating / 5.0, it.click_rate]
    for it in items
], dtype=torch.float32)

# Build and test
item_encoder = MultimodalItemEncoder(
    num_categories=num_categories, num_brands=num_brands, output_dim=64
)

with torch.no_grad():
    item_reprs, gates = item_encoder(
        text_features[:100], image_features[:100],
        categories[:100], brands[:100], metadata[:100]
    )

print(f"Item representation shape: {item_reprs.shape}")
print(f"Gate weights (text, image, meta): {gates.mean(dim=0).numpy().round(3)}")
print(f"Parameters: {sum(p.numel() for p in item_encoder.parameters()):,}")

## 2. Automated Content Tagging

Content tagging automatically assigns labels and attributes to items:
- Category prediction
- Attribute extraction (color, size, material)
- Quality assessment
- Topic assignment

This is typically framed as multi-label classification from item features.

> **\u26a0\ufe0f Common Pitfall:** Automated tagging models need continuous monitoring. As new product types emerge, the model may fail silently. Use confidence thresholds and route low-confidence items for human review.

In [ ]:
class ContentTagger(nn.Module):
    """Multi-label content tagger from item features."""
    
    def __init__(self, input_dim: int, num_tags: int, hidden_dim: int = 64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.tag_heads = nn.Linear(hidden_dim, num_tags)
    
    def forward(self, features: torch.Tensor) -> torch.Tensor:
        h = self.encoder(features)
        return self.tag_heads(h)  # Logits for each tag
    
    def predict_tags(self, features: torch.Tensor, threshold: float = 0.5) -> Tuple[torch.Tensor, torch.Tensor]:
        logits = self.forward(features)
        probs = torch.sigmoid(logits)
        tags = (probs > threshold).float()
        return tags, probs

# Generate synthetic tag data
num_tags = 15
tag_names = [
    'premium', 'budget', 'trending', 'new_arrival', 'bestseller',
    'gift_worthy', 'eco_friendly', 'limited_edition', 'seasonal',
    'high_rated', 'frequently_returned', 'bulk_available',
    'fragile', 'personalized', 'subscription'
]

# Tags correlated with item features
item_input_features = torch.cat([text_features, metadata], dim=-1)  # (5000, 68)
true_tags = torch.zeros(num_items, num_tags)
for i, it in enumerate(items):
    if it.price > 100: true_tags[i, 0] = 1  # premium
    if it.price < 20: true_tags[i, 1] = 1    # budget
    if it.click_rate > 0.3: true_tags[i, 2] = 1  # trending
    if it.num_reviews < 5: true_tags[i, 3] = 1   # new_arrival
    if it.num_reviews > 100: true_tags[i, 4] = 1  # bestseller
    if it.avg_rating > 4.0 and it.price > 30: true_tags[i, 5] = 1  # gift_worthy
    if it.avg_rating > 4.2: true_tags[i, 9] = 1  # high_rated

# Train tagger
tagger = ContentTagger(item_input_features.shape[1], num_tags)
optimizer = torch.optim.Adam(tagger.parameters(), lr=0.005)

for epoch in range(50):
    perm = torch.randperm(num_items)
    for i in range(0, num_items, 256):
        idx = perm[i:i+256]
        logits = tagger(item_input_features[idx])
        loss = F.binary_cross_entropy_with_logits(logits, true_tags[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# Evaluate
with torch.no_grad():
    pred_tags, probs = tagger.predict_tags(item_input_features[:100])

print("Content Tagging Results (first 5 items):")
for i in range(5):
    active_tags = [tag_names[j] for j in range(num_tags) if pred_tags[i, j] > 0]
    print(f"  Item {i} (cat={items[i].category}, ${items[i].price:.0f}): {active_tags}")

## 3. Item Lifecycle Management

Items go through distinct lifecycle phases that affect how they should be recommended:

1. **Cold Start** (new): No behavioral signals, rely on content
2. **Growing** (trending): Increasing engagement, boost exposure
3. **Mature** (stable): Rich behavioral data, standard ranking
4. **Declining**: Decreasing engagement, reduce exposure
5. **Removed**: No longer available

> **\U0001f511 Pro Tip:** Item lifecycle detection should drive recommendation strategy. New items need exploration (more impressions to learn about them). Declining items may indicate quality issues and should be monitored.

In [ ]:
class ItemLifecycleTracker:
    """Tracks and classifies items into lifecycle phases."""
    
    PHASES = ['cold_start', 'growing', 'mature', 'declining', 'removed']
    
    def __init__(
        self,
        cold_start_threshold: int = 10,
        growth_threshold: float = 0.2,
        decline_threshold: float = -0.15,
    ):
        self.cold_start_threshold = cold_start_threshold
        self.growth_threshold = growth_threshold
        self.decline_threshold = decline_threshold
        
        # item_id -> list of (timestamp, engagement_count)
        self.engagement_history: Dict[int, List[Tuple[int, int]]] = defaultdict(list)
    
    def record_engagement(self, item_id: int, timestamp: int, count: int):
        self.engagement_history[item_id].append((timestamp, count))
    
    def classify_phase(self, item_id: int) -> str:
        history = self.engagement_history.get(item_id, [])
        
        if not history:
            return 'cold_start'
        
        total_engagement = sum(c for _, c in history)
        if total_engagement < self.cold_start_threshold:
            return 'cold_start'
        
        if len(history) < 3:
            return 'growing'
        
        # Compute trend from recent vs earlier engagement
        mid = len(history) // 2
        early_avg = np.mean([c for _, c in history[:mid]]) if mid > 0 else 0
        recent_avg = np.mean([c for _, c in history[mid:]])
        
        if early_avg == 0:
            return 'growing'
        
        trend = (recent_avg - early_avg) / (early_avg + 1e-8)
        
        if trend > self.growth_threshold:
            return 'growing'
        elif trend < self.decline_threshold:
            return 'declining'
        else:
            return 'mature'

# Simulate item lifecycles
tracker = ItemLifecycleTracker()
num_sim_items = 500
num_days = 30

# Different lifecycle patterns
for item_id in range(num_sim_items):
    pattern = np.random.choice(['growing', 'mature', 'declining', 'spike'])
    
    for day in range(num_days):
        if pattern == 'growing':
            base = 5 + day * 2
        elif pattern == 'mature':
            base = 30
        elif pattern == 'declining':
            base = max(1, 50 - day * 2)
        else:  # spike
            base = 10 + (40 if 10 <= day <= 15 else 0)
        
        count = max(0, int(base + np.random.normal(0, 3)))
        tracker.record_engagement(item_id, day, count)

# Classify all items
phases = [tracker.classify_phase(i) for i in range(num_sim_items)]
phase_counts = defaultdict(int)
for p in phases:
    phase_counts[p] += 1

print("Item Lifecycle Distribution:")
for phase in ItemLifecycleTracker.PHASES:
    count = phase_counts.get(phase, 0)
    print(f"  {phase}: {count} items ({100*count/num_sim_items:.1f}%)")

## 4. ItemSage: Multi-Task Item Embeddings

**ItemSage** (Vasile et al., Meta, 2023) learns item embeddings that are useful across multiple downstream tasks simultaneously:

- Product search
- Recommendation ranking
- Similar item retrieval
- Category prediction

The key idea: a shared item encoder is trained with multi-task objectives, producing a universal item embedding.

$$\mathcal{L} = \sum_{t=1}^{T} \lambda_t \mathcal{L}_t(f_t(\mathbf{e}_i^{\text{shared}}), y_t)$$

In [ ]:
class ItemSageModel(nn.Module):
    """Multi-task item encoder inspired by ItemSage (Meta, 2023)."""
    
    def __init__(
        self,
        text_dim: int,
        image_dim: int,
        meta_dim: int,
        shared_dim: int = 64,
        num_categories: int = 10,
    ):
        super().__init__()
        
        # Shared encoder
        input_dim = text_dim + image_dim + meta_dim
        self.shared_encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, shared_dim),
            nn.ReLU(),
        )
        
        # Task-specific heads
        self.category_head = nn.Linear(shared_dim, num_categories)
        self.retrieval_head = nn.Linear(shared_dim, shared_dim)  # For contrastive learning
        self.quality_head = nn.Linear(shared_dim, 1)  # Quality score
    
    def encode(self, text: torch.Tensor, image: torch.Tensor, meta: torch.Tensor) -> torch.Tensor:
        x = torch.cat([text, image, meta], dim=-1)
        return self.shared_encoder(x)
    
    def forward(
        self, text: torch.Tensor, image: torch.Tensor, meta: torch.Tensor
    ) -> Dict[str, torch.Tensor]:
        shared = self.encode(text, image, meta)
        
        return {
            'shared_embedding': shared,
            'category_logits': self.category_head(shared),
            'retrieval_embedding': F.normalize(self.retrieval_head(shared), dim=-1),
            'quality_score': torch.sigmoid(self.quality_head(shared)),
        }

# Train ItemSage with multiple objectives
itemsage = ItemSageModel(
    text_dim=64, image_dim=64, meta_dim=4,
    shared_dim=64, num_categories=num_categories
)

# Prepare targets
quality_labels = torch.tensor([it.avg_rating / 5.0 for it in items], dtype=torch.float32)

optimizer = torch.optim.Adam(itemsage.parameters(), lr=0.003)
training_losses = {'category': [], 'retrieval': [], 'quality': []}

for epoch in range(30):
    perm = torch.randperm(num_items)
    epoch_losses = defaultdict(float)
    n_batches = 0
    
    for i in range(0, num_items, 256):
        idx = perm[i:i+256]
        outputs = itemsage(text_features[idx], image_features[idx], metadata[idx])
        
        # Category classification loss
        cat_loss = F.cross_entropy(outputs['category_logits'], categories[idx])
        
        # Contrastive retrieval loss (items in same category should be close)
        batch_cats = categories[idx]
        retr_embs = outputs['retrieval_embedding']
        sim_matrix = retr_embs @ retr_embs.T
        same_cat = (batch_cats.unsqueeze(0) == batch_cats.unsqueeze(1)).float()
        retrieval_loss = F.binary_cross_entropy_with_logits(
            sim_matrix * 5, same_cat  # Scale for sharper supervision
        )
        
        # Quality prediction loss
        quality_loss = F.mse_loss(
            outputs['quality_score'].squeeze(), quality_labels[idx]
        )
        
        # Combined loss
        total_loss = cat_loss + 0.5 * retrieval_loss + 0.3 * quality_loss
        
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        epoch_losses['category'] += cat_loss.item()
        epoch_losses['retrieval'] += retrieval_loss.item()
        epoch_losses['quality'] += quality_loss.item()
        n_batches += 1
    
    for k in epoch_losses:
        training_losses[k].append(epoch_losses[k] / n_batches)

print(f"Final losses - Category: {training_losses['category'][-1]:.4f}, "
      f"Retrieval: {training_losses['retrieval'][-1]:.4f}, "
      f"Quality: {training_losses['quality'][-1]:.4f}")

## 5. Item Graph: Co-purchase, Co-view, Substitutes/Complements

Items form a rich **relational graph** based on user behavior:

- **Co-view**: Items viewed in the same session
- **Co-purchase**: Items bought together
- **Substitutes**: Items that replace each other (same need, different products)
- **Complements**: Items that go well together (phone + case)

These relationships can be used to:
1. Enrich item embeddings via graph propagation
2. Generate "frequently bought together" recommendations
3. Identify substitute products for unavailable items

In [ ]:
class ItemGraph:
    """Item relationship graph with multiple edge types."""
    
    def __init__(self, num_items: int):
        self.num_items = num_items
        self.edges: Dict[str, Dict[int, Dict[int, float]]] = {
            'co_view': defaultdict(lambda: defaultdict(float)),
            'co_purchase': defaultdict(lambda: defaultdict(float)),
            'substitute': defaultdict(lambda: defaultdict(float)),
            'complement': defaultdict(lambda: defaultdict(float)),
        }
    
    def add_edge(self, edge_type: str, item_a: int, item_b: int, weight: float = 1.0):
        self.edges[edge_type][item_a][item_b] += weight
        self.edges[edge_type][item_b][item_a] += weight
    
    def get_neighbors(self, item_id: int, edge_type: str, top_k: int = 10) -> List[Tuple[int, float]]:
        neighbors = self.edges[edge_type].get(item_id, {})
        sorted_neighbors = sorted(neighbors.items(), key=lambda x: -x[1])
        return sorted_neighbors[:top_k]
    
    def to_adjacency_matrix(self, edge_type: str, max_items: int = None) -> torch.Tensor:
        n = max_items or self.num_items
        adj = torch.zeros(n, n)
        for i, neighbors in self.edges[edge_type].items():
            if i >= n:
                continue
            for j, w in neighbors.items():
                if j < n:
                    adj[i, j] = w
        # Normalize
        row_sum = adj.sum(dim=1, keepdim=True).clamp(min=1)
        return adj / row_sum

# Build synthetic item graph from user sessions
graph = ItemGraph(num_items)

# Simulate sessions
num_sessions = 10000
for _ in range(num_sessions):
    # Users tend to view items in same category
    primary_cat = np.random.randint(0, num_categories)
    cat_items = [i for i, it in enumerate(items) if it.category == primary_cat]
    
    if len(cat_items) < 3:
        continue
    
    session_size = np.random.randint(3, min(8, len(cat_items)))
    session_items = np.random.choice(cat_items, size=session_size, replace=False)
    
    # Co-view edges
    for i in range(len(session_items)):
        for j in range(i+1, len(session_items)):
            graph.add_edge('co_view', session_items[i], session_items[j])
    
    # Co-purchase (subset of viewed items)
    if len(session_items) >= 2 and np.random.random() < 0.3:
        purchased = np.random.choice(session_items, size=2, replace=False)
        graph.add_edge('co_purchase', purchased[0], purchased[1])
    
    # Substitutes (same category, similar price)
    for i in range(len(session_items)):
        for j in range(i+1, len(session_items)):
            price_ratio = items[session_items[i]].price / max(items[session_items[j]].price, 0.01)
            if 0.7 < price_ratio < 1.3:
                graph.add_edge('substitute', session_items[i], session_items[j], 0.5)

# Analyze graph
sample_item = 0
print(f"Item {sample_item} (cat={items[sample_item].category}, ${items[sample_item].price:.0f}):")
for edge_type in ['co_view', 'co_purchase', 'substitute']:
    neighbors = graph.get_neighbors(sample_item, edge_type, top_k=5)
    if neighbors:
        neighbor_info = [(n, f"cat={items[n].category},${items[n].price:.0f}", f"w={w:.0f}") 
                        for n, w in neighbors]
        print(f"  {edge_type}: {neighbor_info}")

In [ ]:
# Visualize item graph and lifecycle
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Graph edge distribution
edge_counts = {}
for edge_type in ['co_view', 'co_purchase', 'substitute']:
    total = sum(sum(w for w in neighbors.values()) 
                for neighbors in graph.edges[edge_type].values()) / 2
    edge_counts[edge_type] = int(total)

ax1.bar(edge_counts.keys(), edge_counts.values(), color=['steelblue', 'coral', 'green'], edgecolor='black')
ax1.set_ylabel('Number of Edges', fontsize=12)
ax1.set_title('Item Graph Edge Distribution', fontsize=13, fontweight='bold')
for i, (k, v) in enumerate(edge_counts.items()):
    ax1.text(i, v + 100, f'{v:,}', ha='center', fontsize=10)

# Plot 2: Multi-task training loss
for task, losses in training_losses.items():
    ax2.plot(losses, label=task, linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title('ItemSage Multi-Task Training', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)

plt.tight_layout()
plt.savefig('item_understanding.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Exercises

### \U0001f3cb\ufe0f Exercise 1: Cross-Modal Attention Fusion

Implement a cross-modal attention mechanism where text and image modalities attend to each other.

In [ ]:
# Exercise 1: Cross-Modal Attention
# TODO: Implement cross-attention between text and image modalities

class CrossModalAttention(nn.Module):
    """
    Text attends to image features and vice versa,
    then fuse the cross-attended representations.
    """
    
    def __init__(self, text_dim: int, image_dim: int, output_dim: int):
        super().__init__()
        # TODO: Implement
        # Create text->image attention and image->text attention
        pass
    
    def forward(
        self, text_features: torch.Tensor, image_features: torch.Tensor
    ) -> torch.Tensor:
        # TODO: Implement cross-modal attention fusion
        pass

# Test with synthetic data

### \U0001f3cb\ufe0f Exercise 2: Graph-Enhanced Item Embeddings

Use the item graph to enrich item embeddings through message passing.

In [ ]:
# Exercise 2: Graph-Enhanced Embeddings
# TODO: Implement a simple GNN to enhance item embeddings using the item graph

class GraphEnhancedEmbedding(nn.Module):
    """
    Enhance item embeddings using graph neighborhood aggregation.
    For each item, aggregate neighbor embeddings and combine with own embedding.
    """
    
    def __init__(self, embedding_dim: int, num_layers: int = 2):
        super().__init__()
        # TODO: Implement GNN layers
        pass
    
    def forward(
        self, embeddings: torch.Tensor, adj_matrix: torch.Tensor
    ) -> torch.Tensor:
        # TODO: Apply message passing
        # new_emb = transform(self_emb + aggregate(neighbor_embs))
        pass

# Test: verify that graph-enhanced embeddings for same-category items become more similar

### \U0001f3cb\ufe0f Exercise 3: Item Freshness Scoring

Build a scoring system that adjusts item ranking based on lifecycle phase.

In [ ]:
# Exercise 3: Lifecycle-Aware Scoring
# TODO: Implement a scoring adjustment based on item lifecycle

class LifecycleAwareScorer:
    """
    Adjusts recommendation scores based on item lifecycle:
    - cold_start: boost score to encourage exploration
    - growing: slight boost to ride the trend
    - mature: no adjustment
    - declining: slight penalty
    
    adjusted_score = base_score * lifecycle_multiplier + exploration_bonus
    """
    
    def __init__(self):
        # TODO: Define lifecycle multipliers and exploration bonuses
        pass
    
    def adjust_scores(
        self, base_scores: np.ndarray, lifecycle_phases: List[str]
    ) -> np.ndarray:
        # TODO: Implement score adjustment
        pass

# Test with a mix of items in different phases

## Summary

In this notebook, we built a complete item understanding system:

1. **Multimodal encoding**: Gated fusion of text, image, and metadata features
2. **Content tagging**: Automated multi-label classification for item attributes
3. **Lifecycle management**: Detecting and responding to item lifecycle phases
4. **ItemSage**: Multi-task learning for universal item embeddings
5. **Item graphs**: Capturing co-view, co-purchase, and substitute relationships

### Key References

- Vasile et al. "ItemSage: Learning Product Embeddings for Shopping Recommendations at Pinterest" (KDD, 2023)
- Zhu et al. "Multimodal Pretraining for Product Understanding" (Amazon, 2022)
- McAuley et al. "Inferring Networks of Substitutable and Complementary Products" (KDD, 2015)
- Zheng et al. "A Multi-Task Learning Approach for Item Embeddings" (Meta, 2023)

### Next Steps

In Chapter 8.7, we will explore **Cross-Platform & Cross-Domain** recommendation, transferring user and item representations across different services.